# HW3: прогноз покупок клиента за30дней

Финальная модель: **mlp**, фиксированный seed42,15RFM-признаков.
Модель и параметры зафиксированы в следующей ячейке. Прогнозы разных seed
не усредняются. Внешние данные, бустинг, AutoML, Joint/hurdle и нейронные
товарные ветки не используются. Правило допустимости: семинары1–10.

Источники: [L2/линейная регрессия,2](https://github.com/andriygav/MachineLearningSeminars/blob/master/sem2/main.ipynb),
[MLP,3](https://github.com/andriygav/MachineLearningSeminars/blob/master/sem3/main.ipynb),
[MLP-регрессия/MSE,8](https://github.com/andriygav/MachineLearningSeminars/blob/master/sem8/main.ipynb),
[RandomForest,9](https://github.com/andriygav/MachineLearningSeminars/blob/master/sem9/main.ipynb).

Метрика RMSLE соответствует MSE по log1p(target). После прогноза выполняется
обратное expm1 с ограничением снизу0. Данные берутся только из четырёх
официальных CSV. Очистка/проверка известных таргетов и все функции ниже
представлены открытым исходным кодом; интернет для выполнения не нужен.

При подготовке кода, проверке данных и планировании экспериментов использовался
Codex. Для защиты необходимо понимать очистку, признаки, временной purge,
функцию потерь и причины выбора параметров. Public score не заменяет валидацию.


In [ ]:
RECIPE = {'model': 'mlp', 'profile': 'mlp_base', 'settings': {'hidden_sizes': [128, 64], 'learning_rate': 0.001, 'weight_decay': 0.0001, 'dropout': 0.0, 'batch_size': 1024, 'max_epochs': 200, 'patience': 20, 'min_delta': 0.0001}, 'seed': 42, 'max_seminar': 10}


## Подбор параметров
Процедура использует заранее заданный ручной список параметров MLP, RandomForest и Ridge. Основной подбор идёт на июне–августе2018; September/October используются после фиксации настроек. Дополнительные seed оценивают устойчивость по ошибкам отдельных запусков и не усредняются в прогнозе. Финальный seed42 фиксирован заранее. Вариант mlp_base может служить исходным резервным контролем; факт завершения подбора и его результаты фиксируются отдельно. Полный список и полученные таблицы включаются в delivery ZIP при упаковке. На Kaggle модель автоматически не отправляется.


## Очистка и причинные признаки
Операции используются строго до даты объекта. Все50852 известных ответа восстанавливаются из истории. В модель входят только RFM_COLUMNS, без других возвращаемых служебных статистик.


In [ ]:
"""Cleaning and features with a separate history cutoff for every example."""
from pathlib import Path
import numpy as np
import pandas as pd

RFM_COLUMNS = [
    "recency", "tenure", "n_30", "n_60", "n_90", "n_180",
    "sum_30", "sum_60", "sum_90", "sum_180", "last_amount",
    "mean_amount_180", "median_amount_180", "month_sin", "month_cos",
]
EXTRA_COLUMNS = [
    "median_gap", "gap_cv", "weekly_share", "fortnight_share",
    "monthly_share", "recency_over_gap", "weekday_concentration",
    "clock_concentration", "median_hour", "amount_cv", "amount_iqr_over_median",
]
LOG_COLUMNS = set(RFM_COLUMNS + ["median_gap", "recency_over_gap"]) - {"month_sin", "month_cos"}


def canonical_ids(values):
    return values.astype("string").str.replace(r"\.0$", "", regex=True)


def read_objects(path):
    frame = pd.read_csv(path, dtype={"Id": "string", "customer_id": "string"},
                        parse_dates=["as_of_date", "target_end"])
    frame["customer_id"] = canonical_ids(frame.customer_id)
    if frame[["Id", "customer_id", "as_of_date", "target_end"]].isna().any().any():
        raise ValueError(f"Missing prediction key/date in {path}")
    if not frame.Id.is_unique:
        raise ValueError(f"Duplicate prediction Id in {path}")
    if not frame.as_of_date.eq(frame.as_of_date.dt.normalize()).all():
        raise ValueError("Customer-day features require midnight forecast dates")
    if not frame.target_end.sub(frame.as_of_date).eq(pd.Timedelta(days=30)).all():
        raise ValueError("Every target window must contain exactly 30 days")
    if "spend_30d" in frame and (not np.isfinite(frame.spend_30d).all() or frame.spend_30d.lt(0).any()):
        raise ValueError("Training targets must be finite and nonnegative")
    return frame


def clean_transactions(raw):
    """Positive product purchases, matching the competition's target definition."""
    required = ["event_ts", "order_id", "customer_id", "item_id", "qty", "unit_price"]
    missing = set(required) - set(raw.columns)
    if missing:
        raise ValueError(f"Missing transaction columns: {sorted(missing)}")
    frame = raw.drop_duplicates().copy()
    frame["customer_id"] = canonical_ids(frame.customer_id)
    frame["order_id"] = frame.order_id.astype("string")
    frame["item_id"] = frame.item_id.astype("string")
    frame["event_ts"] = pd.to_datetime(frame.event_ts, errors="coerce")
    for c in ["qty", "unit_price"]:
        frame[c] = pd.to_numeric(frame[c], errors="coerce")
    valid = frame[required].notna().all(axis=1)
    valid &= np.isfinite(frame.qty) & np.isfinite(frame.unit_price)
    valid &= frame.qty.gt(0) & frame.unit_price.gt(0)
    valid &= ~frame.order_id.str.startswith("R", na=True)
    service = frame.item_id.str.fullmatch(r"SVC(?:0[0-9]|1[0-3])", na=True)
    service |= frame.item_id.str.startswith("GFT", na=True)
    frame = frame.loc[valid & ~service].copy()
    frame["amount"] = frame.qty * frame.unit_price
    frame["date"] = frame.event_ts.dt.normalize()
    days = frame.groupby(["customer_id", "date"], sort=True).agg(
        amount=("amount", "sum"), event_ts=("event_ts", "min"),
        n_orders=("order_id", "nunique"), n_products=("item_id", "nunique"),
    ).reset_index()
    days["hour"] = days.event_ts.dt.hour + days.event_ts.dt.minute / 60
    days["dow"] = days.date.dt.dayofweek
    if not np.isclose(days.amount.sum(), frame.amount.sum()):
        raise ValueError("Customer-day aggregation changed total purchase amount")
    stats = {"raw_rows": len(raw), "exact_duplicates": int(raw.duplicated().sum()),
             "clean_rows": len(frame), "customers": int(days.customer_id.nunique()),
             "purchase_days": len(days)}
    return days, stats


class PurchaseHistory:
    def __init__(self, days):
        self.customers = {}
        for cid, group in days.groupby("customer_id"):
            group = group.sort_values("date")
            dates = group.date.to_numpy().astype("datetime64[D]").astype(np.int64)
            a = group.amount.to_numpy(dtype=float)
            self.customers[str(cid)] = (dates, a, np.r_[0.0, np.cumsum(a)],
                                       group.hour.to_numpy(), group.dow.to_numpy(dtype=int))

    def build_features(self, objects):
        rows = []
        empty = (np.array([], dtype=int), np.array([]), np.array([0.0]), np.array([]), np.array([], dtype=int))
        for row in objects.itertuples():
            dates, amounts, cumulative, hours, dows = self.customers.get(str(row.customer_id), empty)
            t = int(np.datetime64(row.as_of_date, "D").astype(np.int64))
            right = int(np.searchsorted(dates, t, side="left"))
            f = {"recency": float(t - dates[right - 1]) if right else 181.0,
                 "tenure": float(t - dates[0]) if right else 0.0}
            for window in [30, 60, 90, 180]:
                left = int(np.searchsorted(dates, t - window, side="left"))
                f[f"n_{window}"] = right - left
                f[f"sum_{window}"] = float(cumulative[right] - cumulative[left])
            left = int(np.searchsorted(dates, t - 180, side="left"))
            a, d, h, dw = amounts[left:right], dates[left:right], hours[left:right], dows[left:right]
            gap = np.diff(d)
            f.update({"last_amount": float(amounts[right - 1]) if right else 0.0,
                      "mean_amount_180": float(a.mean()) if len(a) else 0.0,
                      "median_amount_180": float(np.median(a)) if len(a) else 0.0,
                      "month_sin": float(np.sin(2 * np.pi * row.as_of_date.month / 12)),
                      "month_cos": float(np.cos(2 * np.pi * row.as_of_date.month / 12))})
            f["median_gap"] = float(np.median(gap)) if len(gap) else np.nan
            f["gap_cv"] = float(gap.std() / gap.mean()) if len(gap) >= 3 else np.nan
            f["weekly_share"] = float(((gap >= 6) & (gap <= 8)).mean()) if len(gap) >= 3 else np.nan
            f["fortnight_share"] = float(((gap >= 13) & (gap <= 15)).mean()) if len(gap) >= 3 else np.nan
            f["monthly_share"] = float(((gap >= 26) & (gap <= 32)).mean()) if len(gap) >= 3 else np.nan
            f["recency_over_gap"] = f["recency"] / f["median_gap"] if len(gap) else np.nan
            f["weekday_concentration"] = float(np.bincount(dw, minlength=7).max() / len(dw)) if len(dw) >= 4 else np.nan
            f["clock_concentration"] = float(np.bincount((h // 3).astype(int), minlength=8).max() / len(h)) if len(h) >= 4 else np.nan
            f["median_hour"] = float(np.median(h)) if len(h) >= 4 else np.nan
            f["amount_cv"] = float(a.std() / a.mean()) if len(a) >= 4 else np.nan
            f["amount_iqr_over_median"] = float((np.quantile(a, .75) - np.quantile(a, .25)) / np.median(a)) if len(a) >= 4 else np.nan
            lag_sums = []
            for k in range(6):
                lo = int(np.searchsorted(dates, t - (k + 1) * 30, side="left"))
                hi = int(np.searchsorted(dates, t - k * 30, side="left"))
                lag_sums.append(float(cumulative[hi] - cumulative[lo]))
            f["logmean_prediction"] = float(np.log1p(lag_sums).mean())
            rows.append(f)
        return pd.DataFrame(rows, index=objects.index)

    def reconstruct_targets(self, objects):
        out = np.zeros(len(objects))
        for i, row in enumerate(objects.itertuples()):
            h = self.customers.get(str(row.customer_id))
            if h is None:
                continue
            dates, _, cumulative, _, _ = h
            start = int(np.datetime64(row.as_of_date, "D").astype(np.int64))
            end = int(np.datetime64(row.target_end, "D").astype(np.int64))
            lo, hi = np.searchsorted(dates, [start, end], side="left")
            out[i] = np.round(cumulative[hi] - cumulative[lo], 2)
        return out


def feature_matrix(features, group="rfm"):
    if group not in {"rfm", "extended"}:
        raise ValueError(f"Unknown feature group: {group}")
    columns = RFM_COLUMNS + (EXTRA_COLUMNS if group == "extended" else [])
    x = features[columns].to_numpy(dtype=float).copy()
    for i, column in enumerate(columns):
        if column in LOG_COLUMNS:
            x[:, i] = np.log1p(x[:, i])
    if np.isinf(x).any():
        raise ValueError("Infinite feature value")
    return x


def load_dataset(data_dir):
    data_dir = Path(data_dir)
    raw = pd.read_csv(data_dir / "transactions.csv", dtype={
        "order_id": "string", "customer_id": "string", "item_id": "string",
        "item_name": "string", "region": "string"})
    days, stats = clean_transactions(raw)
    train = read_objects(data_dir / "train.csv")
    test = read_objects(data_dir / "test.csv")
    if set(train.Id) & set(test.Id):
        raise ValueError("Train/test Id overlap")
    history = PurchaseHistory(days)
    rebuilt = history.reconstruct_targets(train)
    error = np.abs(rebuilt - train.spend_30d.to_numpy())
    stats["train_targets_checked"] = len(train)
    stats["max_target_error"] = float(error.max())
    if error.max() > .011:
        raise ValueError(f"Target reconstruction failed: max error {error.max():.4f}")
    return train, test, history, stats


## Обычные регрессоры
MLP имеет один линейный выход. Adam/L2/dropout — обычные параметры сети. Ограничение нормы градиента — численная защита оптимизации. Подбор эпох использует только более раннюю дату, затем модель обучается с нуля.


In [ ]:
import os, random
from copy import deepcopy
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

def make_network(n_features, architecture, task, settings, sequence_shape=None):
    import torch
    assert architecture == 'mlp' and task == 'direct' and sequence_shape is None
    class Network(torch.nn.Module):
        def __init__(self):
            super().__init__()
            layers=[];previous=n_features
            for width in settings['hidden_sizes']:
                layers.extend([torch.nn.Linear(previous,width),torch.nn.ReLU(),torch.nn.Dropout(settings['dropout'])])
                previous=width
            self.body=torch.nn.Sequential(*layers)
            self.output=torch.nn.Linear(previous,1)
        def forward(self,x):return self.output(self.body(x))
    return Network()

def output_prediction(raw,task):
    assert task == 'direct'
    return raw[:,0].clamp_min(0)

def neural_loss(raw,y,w,task):
    import torch
    assert task == 'direct'
    return (((raw[:,0]-torch.log1p(y))**2)*w).sum()/w.sum().clamp_min(1e-12)

class Preprocessor:

    def fit(self, x, w):
        self.imputer = SimpleImputer(strategy='median', add_indicator=False, keep_empty_features=True)
        a = self.imputer.fit_transform(x)
        self.missing = np.flatnonzero(np.isnan(x).any(axis=0))
        a = np.c_[a, np.isnan(x[:, self.missing])]
        self.scaler = StandardScaler().fit(a, sample_weight=w)
        return self

    def transform(self, x):
        return self.apply_state(x, self.state())

    def state(self):
        return {'fill': self.imputer.statistics_.tolist(), 'missing': self.missing.tolist(), 'mean': self.scaler.mean_.tolist(), 'scale': self.scaler.scale_.tolist()}

    @staticmethod
    def apply_state(x, state):
        a = np.where(np.isnan(x), np.asarray(state['fill']), x)
        a = np.c_[a, np.isnan(x[:, state['missing']])]
        return ((a - np.asarray(state['mean'])) / np.asarray(state['scale'])).astype(np.float32)

class DirectMLP:

    def __init__(self, architecture, task, settings, seed=42, device='cpu', threads=2, sequence_shape=None):
        assert architecture == 'mlp' and task == 'direct' and (sequence_shape is None)
        self.architecture, self.task, self.settings = (architecture, task, deepcopy(settings))
        self.seed, self.device, self.threads, self.sequence_shape = (seed, device, threads, sequence_shape)

    def fit(self, x, y, w, validation=None, epochs=None):
        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
        import torch
        if self.device == 'cuda' and (not torch.cuda.is_available()):
            raise RuntimeError('CUDA requested but unavailable')
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        torch.set_num_threads(self.threads)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)
        keep = (w > 0) & True
        x, y, w = (x[keep], y[keep], w[keep])
        if not len(y):
            raise ValueError('Empty neural training task')
        self.preprocessor = Preprocessor().fit(x, w)
        a = self.preprocessor.transform(x)
        self.network = make_network(a.shape[1], 'mlp', 'direct', self.settings, None).to(self.device)
        tx = torch.from_numpy(a).to(self.device)
        ty = torch.as_tensor(y, dtype=torch.float32, device=self.device)
        tw = torch.as_tensor(w, dtype=torch.float32, device=self.device)
        optimizer = torch.optim.Adam(self.network.parameters(), lr=self.settings['learning_rate'], weight_decay=self.settings['weight_decay'])
        if validation is not None:
            vx, vy = validation
            if not len(vy):
                raise ValueError('Empty neural inner validation task')
            va = torch.from_numpy(self.preprocessor.transform(vx)).to(self.device)
        self.history = []
        self.best_epoch = 0
        best = np.inf
        state = None
        stale = 0
        for epoch in range(epochs or self.settings['max_epochs']):
            self.network.train()
            total = 0.0
            mass = 0.0
            for indices in torch.randperm(len(tx), device=self.device).split(self.settings['batch_size']):
                optimizer.zero_grad(set_to_none=True)
                loss = neural_loss(self.network(tx[indices]), ty[indices], tw[indices], 'direct')
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite neural training loss')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.network.parameters(), 10.0)
                optimizer.step()
                total += float(loss.detach()) * len(indices)
                mass += len(indices)
            record = {'epoch': epoch + 1, 'train_loss': total / mass}
            if validation is not None:
                predicted = self._predict_tensor(va)
                score = float(np.sqrt(np.mean((np.log1p(vy) - predicted) ** 2)))
                if not np.isfinite(score):
                    raise FloatingPointError('Non-finite inner metric')
                record['inner_metric'] = score
                if score < best - self.settings['min_delta']:
                    best = score
                    stale = 0
                    self.best_epoch = epoch + 1
                    state = {k: v.detach().cpu().clone() for k, v in self.network.state_dict().items()}
                else:
                    stale += 1
            self.history.append(record)
            if (epoch + 1) % 25 == 0:
                print(f"    epoch {epoch + 1}: loss={record['train_loss']:.5f}", flush=True)
            if validation is not None and stale >= self.settings['patience']:
                break
        if state is not None:
            self.network.load_state_dict(state)
        else:
            self.best_epoch = len(self.history)
        self.network.eval()
        return self

    def _predict_tensor(self, x):
        import torch
        self.network.eval()
        with torch.no_grad():
            result = torch.cat([output_prediction(self.network(chunk), 'direct') for chunk in x.split(4096)])
        return result.cpu().numpy().astype(float)

    def predict(self, x):
        import torch
        a = Preprocessor.apply_state(x, self.loaded_preprocessing) if hasattr(self, 'loaded_preprocessing') else self.preprocessor.transform(x)
        result = self._predict_tensor(torch.from_numpy(a).to(self.device))
        if not np.isfinite(result).all():
            raise FloatingPointError('Non-finite prediction')
        return result

    def save(self, path):
        import torch
        torch.save({'format': 1, 'architecture': 'mlp', 'task': 'direct', 'settings': self.settings, 'sequence_shape': None, 'seed': self.seed, 'preprocessing': self.preprocessor.state(), 'state_dict': {k: v.detach().cpu() for k, v in self.network.state_dict().items()}}, path)

    @classmethod
    def load(cls, path, device='cpu', threads=2):
        import torch
        state = torch.load(path, map_location='cpu', weights_only=True)
        if state['format'] != 1:
            raise ValueError('Unknown neural checkpoint')
        model = cls(state['architecture'], state['task'], state['settings'], state['seed'], device, threads, state['sequence_shape'])
        model.loaded_preprocessing = state['preprocessing']
        model.network = make_network(len(state['preprocessing']['mean']), model.architecture, model.task, model.settings, model.sequence_shape).to(device)
        model.network.load_state_dict(state['state_dict'])
        model.network.eval()
        return model


## Данные и параметры запуска
Укажите путь к четырём официальным CSV. При наличии rfm_cache.npz его значения сверяются с расчётом из сырой истории. Для точного воспроизведения исходного запуска предпочтительны прежние версии библиотек и V100; CPU/GPU могут давать разные траектории обучения.


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
THREADS = 2
DATA_DIR = Path('data/raw')  # Put the four official CSV files here.
train, test, history, checks = load_dataset(DATA_DIR)
objects = pd.concat([train.assign(is_test=False),test.assign(is_test=True)],ignore_index=True)
objects = objects.sort_values(['as_of_date','customer_id']).reset_index(drop=True)
features = history.build_features(objects)
X = feature_matrix(features, 'rfm').astype(np.float32)
assert X.shape[1] == 15 and np.isfinite(X).all()
# Optional exact cache, included in the delivery ZIP; independently check it against raw history.
if Path('rfm_cache.npz').exists():
    with np.load('rfm_cache.npz',allow_pickle=False) as cached:
        order = pd.Index(cached['Id']).get_indexer(objects.Id.to_numpy(str))
        assert (order >= 0).all()
        exact = cached['X'][order]
    np.testing.assert_allclose(X,exact,rtol=2e-6,atol=2e-6)
    X = exact
print(checks)
print('Model:', RECIPE['model'], 'seed:', 42, 'features:', RFM_COLUMNS)
print('Settings:', RECIPE['settings'])

def fit_predict(cutoff, prediction_indices):
    cutoff = pd.Timestamp(cutoff)
    tr = np.flatnonzero((~objects.is_test & objects.target_end.le(cutoff)).to_numpy())
    y = objects.iloc[tr].spend_30d.to_numpy(float)
    settings = RECIPE['settings']
    if RECIPE['model'] == 'mlp':
        inner = objects.loc[~objects.is_test & objects.target_end.le(cutoff),'as_of_date'].max()
        ti = np.flatnonzero((~objects.is_test & objects.target_end.le(inner)).to_numpy())
        iv = np.flatnonzero((~objects.is_test & objects.as_of_date.eq(inner)).to_numpy())
        selector = DirectMLP('mlp','direct',settings,42,DEVICE,THREADS)
        selector.fit(X[ti],objects.iloc[ti].spend_30d.to_numpy(float),np.ones(len(ti)),
                     validation=(X[iv],objects.iloc[iv].spend_30d.to_numpy(float)))
        epochs = selector.best_epoch
        del selector
        model = DirectMLP('mlp','direct',settings,42,DEVICE,THREADS)
        model.fit(X[tr],y,np.ones(len(tr)),epochs=epochs)
        assert model.network.output.out_features == 1
        z = model.predict(X[prediction_indices])
    else:
        preprocessing = Preprocessor().fit(X[tr],np.ones(len(tr)))
        a = preprocessing.transform(X[tr])
        if RECIPE['model'] == 'ridge': model = Ridge(alpha=settings['alpha'])
        elif RECIPE['model'] == 'forest': model = RandomForestRegressor(**settings,random_state=42,n_jobs=THREADS)
        else: raise ValueError('Unapproved model')
        model.fit(a,np.log1p(y),sample_weight=np.ones(len(tr)))
        z = np.maximum(model.predict(preprocessing.transform(X[prediction_indices])),0)
    assert np.isfinite(z).all() and (z >= 0).all()
    return z


## Временная проверка
Train: target_end<=cutoff. Valid: as_of_date=cutoff. Число эпох выбирается на более ранней внутренней дате. Сентябрь/октябрь уже просматривались, оценка не является новым untouched holdout.


Зафиксированные результаты исходного запуска:

| Date | RMSLE |
|---|---:|
| September2018 | 2.631250 |
| October2018 | 2.689396 |
| Pooled Sep-Oct | 2.661253 |


In [ ]:
scores=[]
for date in ['2018-09-01','2018-10-01']:
    vi=np.flatnonzero((~objects.is_test & objects.as_of_date.eq(date)).to_numpy())
    z=fit_predict(date,vi)
    error=(z-np.log1p(objects.iloc[vi].spend_30d.to_numpy(float)))**2
    scores.append({'date':date,'n':len(vi),'squared_log_error':float(error.sum()),'RMSLE':float(np.sqrt(error.mean()))})
display(pd.DataFrame(scores))
print('Pooled RMSLE:', np.sqrt(sum(r['squared_log_error'] for r in scores)/sum(r['n'] for r in scores)))


## Финальный рефит
Одна модель с seed42 обучается на всех доступных к2018-11-01 метках. CSV содержит все3188Id в порядке sample. Эта ячейка создаёт файл и не отправляет его в Kaggle.


In [ ]:
vi=np.flatnonzero(objects.is_test.to_numpy())
z=fit_predict('2018-11-01',vi)
predictions=pd.DataFrame({'Id':objects.iloc[vi].Id.to_numpy(str),'spend_30d':np.maximum(np.expm1(z),0)})
sample=pd.read_csv(DATA_DIR/'sample_submission.csv',dtype={'Id':str})
assert predictions.Id.is_unique and set(predictions.Id)==set(sample.Id)
predictions=predictions.set_index('Id').loc[sample.Id].reset_index()
assert np.isfinite(predictions.spend_30d).all() and predictions.spend_30d.ge(0).all()
predictions.to_csv('submission_reproduced.csv',index=False)
print('Created submission_reproduced.csv:',len(predictions),'rows. No automatic Kaggle upload.')
